# Duygu ve Aciliyet Göstergeleri ile Oltalama Tespiti
**Görkem Poyrazoğlu — Begüm Düger**

**Bu notebook henüz Kaggle verisiyle çalıştırılmamıştır. Çıktılar bilinçli olarak boştur.**

Kaggle'da **Add Input / Add Data** ile yalnızca [naserabdullahalam/phishing-email-dataset](https://www.kaggle.com/datasets/naserabdullahalam/phishing-email-dataset) veri setini bağlayın. Sonra **Run All** çalıştırın. CPU yeterlidir; bağlı veri ile internet gerekmez.

Pozitif etiketin kapsamı veri kaynağından doğrulanmalıdır. Sözcük listeleri duygu için vekil göstergelerdir; doğrulanmış psikolojik duygu etiketleri değildir. Yapay veya alternatif veriye otomatik geçiş yapılmaz.

## 1. Tekrarlanabilir deney modülü
Kaynak kod depoda tutulur; bu hücre veya kod satırları bildiriye/sunuma kopyalanmaz. Kod aşağıda ayrı bir Python modülüne yazılır.

In [ ]:
from pathlib import Path
source = '"""Reproducible experiment for naserabdullahalam/phishing-email-dataset.\n\nThis program never substitutes another corpus or manufactures research results.\nThe lexicons below are hand-written proxies, not ground-truth emotion labels.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport html\nimport importlib.metadata\nimport json\nimport platform\nimport re\nimport sys\nimport time\nimport unicodedata\nimport warnings\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom sklearn.exceptions import ConvergenceWarning\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (accuracy_score, average_precision_score,\n                             balanced_accuracy_score, classification_report,\n                             confusion_matrix, f1_score, precision_score,\n                             recall_score, roc_auc_score)\nfrom sklearn.model_selection import StratifiedGroupKFold\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_limits\n\nDATASET_HANDLE = "naserabdullahalam/phishing-email-dataset"\nDATASET_URL = f"https://www.kaggle.com/datasets/{DATASET_HANDLE}"\nEXPECTED_FILENAME = "phishing_email.csv"\nSEED = 42\nMODEL_NAMES = ("A_text_only", "B_cues_only", "C_text_and_cues")\nC_GRID = (0.1, 1.0, 10.0)\n\n# Frozen before seeing the evaluation data. Keep these lists unchanged during a run.\n# Matching words does not establish the sender\'s intent or the recipient\'s emotion.\nLEXICONS = {\n    "urgency": (\n        "urgent", "urgently", "immediately", "as soon as possible", "asap",\n        "act now", "respond now", "right now", "today", "deadline", "expires",\n        "expiring", "last chance", "limited time", "within 24 hours",\n        "within 48 hours", "final notice", "time sensitive", "without delay",\n    ),\n    "fear_threat": (\n        "suspended", "suspension", "blocked", "terminated", "termination",\n        "unauthorized", "fraud", "compromised", "security breach", "penalty",\n        "legal action", "account closure", "account locked", "lose access",\n        "identity theft", "warning", "risk", "threat", "danger",\n    ),\n    "reward": (\n        "winner", "won", "prize", "reward", "congratulations", "bonus",\n        "lottery", "jackpot", "free", "exclusive offer", "cash reward",\n        "claim your", "inheritance", "million dollars", "beneficiary",\n        "discount", "gift", "guaranteed", "special offer",\n    ),\n    "trust_authority": (\n        "official", "verified", "secure", "trusted", "security team",\n        "support team", "customer support", "administrator", "bank",\n        "government", "verify your account", "confirm your identity",\n        "account verification", "security department", "authorized",\n    ),\n}\n\n\ndef _compile_phrases(phrases: tuple[str, ...]) -> re.Pattern:\n    alternatives = []\n    for phrase in sorted(phrases, key=len, reverse=True):\n        alternatives.append(r"[\\s_-]+".join(re.escape(p) for p in phrase.split()))\n    return re.compile(r"(?<!\\w)(?:" + "|".join(alternatives) + r")(?!\\w)", re.I)\n\n\nPATTERNS = {name: _compile_phrases(words) for name, words in LEXICONS.items()}\nURL_PATTERN = re.compile(r"(?:https?://|www\\.)[^\\s<>]+", re.I)\nEMAIL_PATTERN = re.compile(r"\\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}\\b")\nFEATURE_NAMES = [f"{name}_{stat}" for name in LEXICONS\n                 for stat in ("present", "per_100_words", "log_count")]\nFEATURE_NAMES += ["urgency_and_fear", "urgency_and_reward"]\n\n\ndef write_json(path: Path, obj: Any) -> None:\n    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef clean_text(value: str, max_chars: int = 20_000) -> str:\n    """Deterministic text-only preprocessing; never visits URLs or executes HTML."""\n    text = unicodedata.normalize("NFKC", html.unescape(str(value)))\n    text = re.sub(r"<(script|style)\\b[^>]*>.*?</\\1\\s*>", " ", text, flags=re.I | re.S)\n    text = re.sub(r"<[^>]+>", " ", text)\n    text = URL_PATTERN.sub(" urltoken ", text)\n    text = EMAIL_PATTERN.sub(" emailtoken ", text)\n    text = re.sub(r"\\s+", " ", text).strip().casefold()\n    return text[:max_chars].strip()\n\n\ndef template_hash(text: str) -> str:\n    """Conservative heuristic, not a semantic near-duplicate detector.\n\n    URL/email placeholders were already introduced by clean_text. Number and\n    punctuation variants share a group. Paraphrased campaigns can still leak.\n    """\n    value = re.sub(r"\\d+", " numbertoken ", text)\n    value = re.sub(r"[^\\w\\s]", " ", value)\n    value = re.sub(r"\\s+", " ", value).strip()\n    return hashlib.sha256(value.encode("utf-8")).hexdigest()\n\n\ndef cue_matrix(texts: list[str]) -> np.ndarray:\n    features = np.zeros((len(texts), len(FEATURE_NAMES)), dtype=np.float64)\n    for row, text in enumerate(texts):\n        count_words = max(1, len(re.findall(r"\\b\\w+\\b", text)))\n        counts = [len(pattern.findall(text)) for pattern in PATTERNS.values()]\n        for index, count in enumerate(counts):\n            features[row, index * 3:index * 3 + 3] = (\n                float(count > 0), 100.0 * count / count_words, np.log1p(count))\n        features[row, -2:] = [float(counts[0] > 0 and counts[1] > 0),\n                             float(counts[0] > 0 and counts[2] > 0)]\n    return features\n\n\ndef prepare_frame(raw: pd.DataFrame, max_chars: int = 20_000) -> tuple[pd.DataFrame, dict]:\n    if not {"text_combined", "label"}.issubset(raw.columns):\n        raise ValueError("Beklenen sütunlar: text_combined, label. Başka veri seti kullanılmayacak.")\n    frame = raw[["text_combined", "label"]].copy()\n    frame["source_row"] = np.arange(len(frame))\n    audit = {"raw_rows": len(frame), "input_columns": list(raw.columns),\n             "max_chars_after_cleaning": max_chars}\n    nulls = frame["text_combined"].isna() | frame["label"].isna()\n    audit["missing_text_or_label_rows"] = int(nulls.sum())\n    frame = frame.loc[~nulls].copy()\n    numeric_labels = pd.to_numeric(frame["label"], errors="coerce")\n    invalid = numeric_labels.isna() | ~numeric_labels.isin([0, 1])\n    if invalid.any():\n        raise ValueError(f"0/1 dışında veya çözülemeyen {int(invalid.sum())} etiket var; otomatik dönüştürülmedi.")\n    frame["label"] = numeric_labels.astype(int)\n    audit["nonmissing_input_class_counts"] = {str(k): int(v) for k, v in frame.label.value_counts().sort_index().items()}\n    audit["raw_rows_longer_than_char_limit"] = int((frame.text_combined.astype(str).str.len() > max_chars).sum())\n    frame["text"] = frame.text_combined.astype(str).map(lambda s: clean_text(s, max_chars))\n    empty = ~frame.text.str.contains(r"\\w", regex=True)\n    audit["empty_after_cleaning_rows"] = int(empty.sum())\n    frame = frame.loc[~empty].copy()\n    frame["text_hash"] = frame.text.map(lambda s: hashlib.sha256(s.encode("utf-8")).hexdigest())\n    label_counts = frame.groupby("text_hash").label.nunique()\n    conflicting_hashes = set(label_counts[label_counts > 1].index)\n    conflicting = frame.text_hash.isin(conflicting_hashes)\n    audit["conflicting_text_groups"] = len(conflicting_hashes)\n    audit["conflicting_rows_removed"] = int(conflicting.sum())\n    frame = frame.loc[~conflicting].copy()\n    audit["duplicate_rows_removed"] = int(frame.duplicated("text_hash").sum())\n    frame = frame.drop_duplicates("text_hash").reset_index(drop=True)\n    frame["group_hash"] = frame.text.map(template_hash)\n    audit["retained_rows"] = len(frame)\n    audit["retained_class_counts"] = {str(k): int(v) for k, v in frame.label.value_counts().sort_index().items()}\n    audit["template_groups"] = int(frame.group_hash.nunique())\n    audit["groups_with_both_labels_retained_together"] = int((frame.groupby("group_hash").label.nunique() > 1).sum())\n    audit["language_detection_performed"] = False\n    audit["emotion_ground_truth_available"] = False\n    return frame[["source_row", "text", "label", "text_hash", "group_hash"]], audit\n\n\ndef make_splits(frame: pd.DataFrame, seed: int = SEED) -> dict[str, np.ndarray]:\n    if set(frame.label.unique()) != {0, 1}:\n        raise ValueError("Her iki sınıf da gerekli.")\n    groups_per_class = frame.groupby("label").group_hash.nunique()\n    if groups_per_class.min() < 10:\n        raise ValueError("Her sınıfta en az 10 ayrı metin/şablon grubu gerekli.")\n    y, groups = frame.label.to_numpy(), frame.group_hash.to_numpy()\n    outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)\n    dev, test = next(outer.split(np.zeros(len(y)), y, groups))\n    inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed + 1)\n    train_local, val_local = next(inner.split(np.zeros(len(dev)), y[dev], groups[dev]))\n    partitions = {"train": dev[train_local], "validation": dev[val_local], "test": test}\n    names = list(partitions)\n    for name, indices in partitions.items():\n        if set(y[indices]) != {0, 1}:\n            raise ValueError(f"{name} bölmesinde iki sınıf da bulunmuyor; deney durduruldu.")\n    for i, left in enumerate(names):\n        for right in names[i + 1:]:\n            assert not (set(groups[partitions[left]]) & set(groups[partitions[right]])), "Group leakage"\n            assert not (set(frame.text_hash.iloc[partitions[left]]) & set(frame.text_hash.iloc[partitions[right]])), "Text leakage"\n    return partitions\n\n\ndef locate_csv(explicit: str | None = None, allow_download: bool = False) -> tuple[Path, str]:\n    if explicit:\n        path = Path(explicit).expanduser().resolve()\n        if not path.is_file():\n            raise FileNotFoundError(path)\n        if path.name != EXPECTED_FILENAME:\n            raise ValueError(f"Dosya adı {EXPECTED_FILENAME} olmalı; farklı dosya otomatik kabul edilmiyor.")\n        return path, "user_supplied_local_file_provenance_not_independently_verified"\n    roots = (Path("/kaggle/input"), Path("data"), Path.cwd())\n    candidates = set()\n    for root in roots:\n        if root.is_dir():\n            candidates.update(p.resolve() for p in root.rglob(EXPECTED_FILENAME) if p.is_file())\n    if len(candidates) == 1:\n        path = next(iter(candidates))\n        return path, "mounted_file_schema_checked_origin_requires_dataset_attachment_check"\n    if len(candidates) > 1:\n        raise ValueError("Birden çok phishing_email.csv bulundu. --csv ile tam dosya yolunu belirtin.")\n    if not allow_download:\n        raise FileNotFoundError("phishing_email.csv yok. Tam olarak belirtilen Kaggle veri setini ekleyin veya --download kullanın.")\n    try:\n        import kagglehub\n    except ImportError as exc:\n        raise RuntimeError("İndirme için: python -m pip install kagglehub") from exc\n    try:\n        directory = Path(kagglehub.dataset_download(DATASET_HANDLE))\n    except Exception as exc:\n        raise RuntimeError(f"{DATASET_HANDLE} indirilemedi. Başka veri veya sahte veri kullanılmadı. Asıl hata: {exc}") from exc\n    matches = list(directory.rglob(EXPECTED_FILENAME))\n    if len(matches) != 1:\n        raise FileNotFoundError(f"Kaggle paketinde tek bir {EXPECTED_FILENAME} bekleniyordu; {len(matches)} bulundu.")\n    return matches[0].resolve(), "downloaded_by_kagglehub_exact_dataset_handle"\n\n\ndef fit_representations(train_texts: list[str], eval_texts: list[str], max_features: int):\n    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=1.0,\n                               max_features=max_features, sublinear_tf=True,\n                               lowercase=False, dtype=np.float64)\n    text_train = vectorizer.fit_transform(train_texts)\n    text_eval = vectorizer.transform(eval_texts)\n    cue_scaler = StandardScaler()\n    cue_train = sparse.csr_matrix(cue_scaler.fit_transform(cue_matrix(train_texts)))\n    cue_eval = sparse.csr_matrix(cue_scaler.transform(cue_matrix(eval_texts)))\n    train_matrices = (text_train, cue_train, sparse.hstack([text_train, cue_train], format="csr"))\n    eval_matrices = (text_eval, cue_eval, sparse.hstack([text_eval, cue_eval], format="csr"))\n    return vectorizer, cue_scaler, train_matrices, eval_matrices\n\n\ndef train_classifier(X: sparse.spmatrix, y: np.ndarray, C: float, seed: int):\n    model = LogisticRegression(C=C, solver="liblinear", class_weight="balanced",\n                               max_iter=3000, tol=1e-4, random_state=seed)\n    with warnings.catch_warnings(record=True) as caught:\n        warnings.simplefilter("always", ConvergenceWarning)\n        model.fit(X, y)\n    if any(issubclass(item.category, ConvergenceWarning) for item in caught):\n        raise RuntimeError("Model yakınsamadı. Sonuç yazılmadı; max_iter/ölçeklemeyi gözden geçirin.")\n    return model\n\n\ndef calculate_metrics(y: np.ndarray, probability: np.ndarray) -> dict:\n    prediction = (probability >= 0.5).astype(int)\n    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()\n    return {\n        "accuracy": float(accuracy_score(y, prediction)),\n        "balanced_accuracy": float(balanced_accuracy_score(y, prediction)),\n        "precision_positive": float(precision_score(y, prediction, zero_division=0)),\n        "recall_positive": float(recall_score(y, prediction, zero_division=0)),\n        "f1_positive": float(f1_score(y, prediction, zero_division=0)),\n        "f1_macro": float(f1_score(y, prediction, average="macro", zero_division=0)),\n        "roc_auc": float(roc_auc_score(y, probability)),\n        "average_precision_ap": float(average_precision_score(y, probability)),\n        "false_positive_rate": float(fp / (fp + tn)) if fp + tn else None,\n        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),\n        "test_rows": int(len(y)), "threshold": 0.5,\n    }\n\n\ndef paired_group_bootstrap(y: np.ndarray, pred_a: np.ndarray, pred_c: np.ndarray,\n                           groups: np.ndarray, n_boot: int, seed: int) -> dict:\n    """Resample template groups, paired across A and C; report positive F1 delta."""\n    _, inv = np.unique(groups, return_inverse=True)\n    number_groups = int(inv.max()) + 1\n    counts = np.zeros((number_groups, 6), dtype=np.int64)\n    for offset, pred in ((0, pred_a), (3, pred_c)):\n        for j, mask in enumerate(((y == 1) & (pred == 1),\n                                  (y == 0) & (pred == 1),\n                                  (y == 1) & (pred == 0))):\n            counts[:, offset + j] = np.bincount(inv, weights=mask, minlength=number_groups).astype(np.int64)\n    rng = np.random.default_rng(seed)\n    deltas = []\n    for _ in range(n_boot):\n        totals = counts[rng.integers(0, number_groups, number_groups)].sum(axis=0)\n        tpa, fpa, fna, tpc, fpc, fnc = totals\n        da, dc = 2 * tpa + fpa + fna, 2 * tpc + fpc + fnc\n        if da and dc:\n            deltas.append(2 * tpc / dc - 2 * tpa / da)\n    observed = float(f1_score(y, pred_c, zero_division=0) - f1_score(y, pred_a, zero_division=0))\n    interval = np.quantile(deltas, [0.025, 0.975]).tolist() if deltas else [None, None]\n    return {"quantity": "positive_F1_C_minus_A", "observed_delta": observed,\n            "ci_95_percentile": interval, "bootstrap_replicates": n_boot,\n            "valid_replicates": len(deltas), "resampling_unit": "heuristic_template_group",\n            "seed": seed,\n            "caution": "Conditional on this split and fixed models; not an external-validity or causality test."}\n\n\ndef make_figures(results_dir: Path, metrics: pd.DataFrame, y: np.ndarray,\n                 predictions: pd.DataFrame) -> None:\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    fig, ax = plt.subplots(figsize=(8, 4.5))\n    ax.bar(metrics.model, metrics.f1_positive)\n    ax.set_ylim(0, 1.05)\n    ax.set_ylabel("Positive-class F1")\n    ax.set_title("Held-out test: model comparison")\n    for index, value in enumerate(metrics.f1_positive):\n        ax.text(index, value + 0.015, f"{value:.4f}", ha="center")\n    fig.tight_layout()\n    fig.savefig(results_dir / "f1_comparison.png", dpi=180)\n    plt.close(fig)\n    for model in MODEL_NAMES:\n        matrix = confusion_matrix(y, predictions[f"{model}_prediction"], labels=[0, 1])\n        fig, ax = plt.subplots(figsize=(5, 4.5))\n        ax.imshow(matrix)\n        for (row, col), value in np.ndenumerate(matrix):\n            ax.text(col, row, str(value), ha="center", va="center",\n                    bbox=dict(boxstyle="round", alpha=0.85))\n        ax.set_xticks([0, 1], ["Label 0", "Label 1"])\n        ax.set_yticks([0, 1], ["Label 0", "Label 1"])\n        ax.set_xlabel("Predicted label")\n        ax.set_ylabel("Dataset label")\n        ax.set_title(model)\n        fig.tight_layout()\n        fig.savefig(results_dir / f"confusion_{model}.png", dpi=180)\n        plt.close(fig)\n\n\ndef run_experiment(csv_path: Path, output: Path, source_mode: str,\n                   seed: int = SEED, max_features: int = 50_000,\n                   max_chars: int = 20_000, bootstrap: int = 1000,\n                   save_models: bool = False) -> dict:\n    if output.exists() and any(output.iterdir()):\n        raise FileExistsError(f"Çıktı dizini boş değil: {output}. Yeni bir dizin kullanın; eski sonuçlar ezilmeyecek.")\n    output.mkdir(parents=True, exist_ok=True)\n    started = time.perf_counter()\n    manifest = {\n        "status": "started_not_completed", "started_utc": datetime.now(timezone.utc).isoformat(),\n        "dataset_handle": DATASET_HANDLE, "dataset_url": DATASET_URL,\n        "input_file_name": csv_path.name, "input_file_bytes": csv_path.stat().st_size,\n        "input_sha256": sha256_file(csv_path), "source_mode": source_mode,\n        "python": sys.version, "platform": platform.platform(),\n        "dependencies": {name: importlib.metadata.version(name) for name in\n                         ("numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "threadpoolctl")},\n        "configuration": {"seed": seed, "C_grid": list(C_GRID), "max_features": max_features,\n                          "max_chars": max_chars, "test_threshold": 0.5,\n                          "class_weight": "balanced", "ngram_range": [1, 2],\n                          "selection_metric": "validation_macro_F1",\n                          "split": "first fold of group-stratified 5-fold outer, 4-fold inner",\n                          "bootstrap_replicates": bootstrap},\n        "limitations": [\n            "Positive class may pool spam, scams and phishing; do not claim pure credential-phishing accuracy.",\n            "Emotion cues are investigator-defined English lexicons; no emotion ground truth or causality claim.",\n            "Template grouping cannot remove all semantic near duplicates or source-domain leakage.",\n            "No independent modern, Turkish, temporal or source-held-out evaluation is performed.",\n            "A single fixed grouped split is used; results are split-specific.",\n            "Input text is truncated after deterministic normalization at the configured character limit.",\n        ],\n    }\n    write_json(output / "run_manifest.json", manifest)\n    try:\n        raw = pd.read_csv(csv_path, dtype={"text_combined": "string", "label": "string"}, on_bad_lines="error")\n        frame, audit = prepare_frame(raw, max_chars)\n        del raw\n        write_json(output / "data_audit.json", audit)\n        if len(frame) < 100:\n            raise ValueError("100\'den az kullanılabilir kayıt: araştırma deneyi çalıştırılmadı.")\n        splits = make_splits(frame, seed)\n        split_table = frame[["source_row", "text_hash", "group_hash", "label"]].copy()\n        split_table["split"] = ""\n        for name, indices in splits.items():\n            split_table.loc[indices, "split"] = name\n        split_table.to_csv(output / "split_manifest.csv", index=False)\n        manifest["split_counts"] = {name: {"rows": len(indices),\n            "label_0": int((frame.label.iloc[indices] == 0).sum()),\n            "label_1": int((frame.label.iloc[indices] == 1).sum())} for name, indices in splits.items()}\n        manifest["exact_hash_overlap_between_splits"] = 0\n        manifest["heuristic_group_overlap_between_splits"] = 0\n        print("Data audit:", audit, flush=True)\n        print("Split counts:", manifest["split_counts"], flush=True)\n        train, validation, test = splits["train"], splits["validation"], splits["test"]\n        y = frame.label.to_numpy()\n        _, _, X_train, X_val = fit_representations(frame.text.iloc[train].tolist(),\n                                                   frame.text.iloc[validation].tolist(), max_features)\n        tuning_rows = []\n        best_C = {}\n        with threadpool_limits(limits=1):\n            for index, name in enumerate(MODEL_NAMES):\n                candidates = []\n                for C in C_GRID:\n                    model = train_classifier(X_train[index], y[train], C, seed)\n                    prediction = (model.predict_proba(X_val[index])[:, 1] >= 0.5).astype(int)\n                    score = float(f1_score(y[validation], prediction, average="macro", zero_division=0))\n                    candidates.append((score, C))\n                    tuning_rows.append({"model": name, "C": C, "validation_macro_F1": score})\n                # Stable tie-break: smaller regularization inverse C wins.\n                best_C[name] = sorted(candidates, key=lambda item: (-item[0], item[1]))[0][1]\n                print(name, "selected C:", best_C[name], flush=True)\n        pd.DataFrame(tuning_rows).to_csv(output / "validation_scores.csv", index=False)\n        del X_train, X_val\n        development = np.sort(np.concatenate([train, validation]))\n        vectorizer, scaler, X_dev, X_test = fit_representations(\n            frame.text.iloc[development].tolist(), frame.text.iloc[test].tolist(), max_features)\n        predictions = frame.loc[test, ["source_row", "text_hash", "group_hash", "label"]].reset_index(drop=True)\n        rows, report = [], {}\n        models = {}\n        with threadpool_limits(limits=1):\n            for index, name in enumerate(MODEL_NAMES):\n                model = train_classifier(X_dev[index], y[development], best_C[name], seed)\n                probability = model.predict_proba(X_test[index])[:, 1]\n                prediction = (probability >= 0.5).astype(int)\n                models[name] = model\n                predictions[f"{name}_probability"] = probability\n                predictions[f"{name}_prediction"] = prediction\n                measured = calculate_metrics(y[test], probability)\n                rows.append({"model": name, "selected_C": best_C[name], **measured})\n                report[name] = classification_report(y[test], prediction, labels=[0, 1],\n                                                     target_names=["dataset_label_0", "dataset_label_1"],\n                                                     output_dict=True, zero_division=0)\n                print(name, measured, flush=True)\n        metrics = pd.DataFrame(rows)\n        metrics.to_csv(output / "metrics.csv", index=False)\n        write_json(output / "metrics.json", rows)\n        write_json(output / "classification_reports.json", report)\n        predictions.to_csv(output / "test_predictions.csv", index=False)\n        difference = paired_group_bootstrap(y[test], predictions.A_text_only_prediction.to_numpy(),\n            predictions.C_text_and_cues_prediction.to_numpy(), predictions.group_hash.to_numpy(), bootstrap, seed + 2)\n        write_json(output / "paired_f1_comparison.json", difference)\n        cues_test = cue_matrix(frame.text.iloc[test].tolist())\n        descriptors = []\n        for index, name in enumerate(LEXICONS):\n            for label in (0, 1):\n                subset = cues_test[y[test] == label]\n                descriptors.append({"cue": name, "dataset_label": label, "test_rows": len(subset),\n                                    "presence_rate": float(subset[:, index * 3].mean()),\n                                    "mean_count_per_100_words": float(subset[:, index * 3 + 1].mean())})\n        pd.DataFrame(descriptors).to_csv(output / "cue_descriptives_test.csv", index=False)\n        pd.DataFrame({"feature": FEATURE_NAMES, "coefficient": models["B_cues_only"].coef_[0]}).to_csv(\n            output / "cue_model_coefficients.csv", index=False)\n        if save_models:\n            import joblib\n            joblib.dump({"vectorizer": vectorizer, "cue_scaler": scaler, "models": models,\n                         "feature_names": FEATURE_NAMES, "max_chars": max_chars,\n                         "warning": "Do not load untrusted joblib/pickle files."}, output / "models.joblib")\n        make_figures(output, metrics, y[test], predictions)\n        manifest.update({"status": "completed", "finished_utc": datetime.now(timezone.utc).isoformat(),\n                         "elapsed_seconds": time.perf_counter() - started, "selected_C": best_C,\n                         "tfidf_feature_count": len(vectorizer.get_feature_names_out()),\n                         "cue_feature_count": len(FEATURE_NAMES)})\n        write_json(output / "run_manifest.json", manifest)\n        return {"metrics": rows, "audit": audit, "comparison": difference, "manifest": manifest}\n    except Exception as exc:\n        manifest.update({"status": "failed_not_a_completed_experiment", "error": str(exc),\n                         "elapsed_seconds": time.perf_counter() - started})\n        write_json(output / "run_manifest.json", manifest)\n        raise\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--csv", help="Exact phishing_email.csv path")\n    parser.add_argument("--download", action="store_true", help="Download ONLY the specified Kaggle dataset")\n    parser.add_argument("--output", default="results/real_run")\n    parser.add_argument("--seed", type=int, default=SEED)\n    parser.add_argument("--max-features", type=int, default=50_000)\n    parser.add_argument("--max-chars", type=int, default=20_000)\n    parser.add_argument("--bootstrap", type=int, default=1000)\n    parser.add_argument("--save-models", action="store_true")\n    args = parser.parse_args()\n    if args.max_features < 10 or args.max_chars < 100 or args.bootstrap < 100:\n        parser.error("max-features >= 10, max-chars >= 100, bootstrap >= 100 olmalı.")\n    try:\n        path, mode = locate_csv(args.csv, args.download)\n        run_experiment(path, Path(args.output), mode, args.seed, args.max_features,\n                       args.max_chars, args.bootstrap, args.save_models)\n    except Exception as exc:\n        print(f"DENEY TAMAMLANMADI: {exc}", file=sys.stderr)\n        return 1\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'
Path("experiment.py").write_text(source, encoding="utf-8")
print("Deney modülü hazır. Henüz veri eğitimi yapılmadı.")

## 2. Gerçek Kaggle girdisini bul
Birden fazla aynı adlı CSV varsa otomatik seçim yapılmaz. Veri setini değiştirmek veya küçük bir uydurma veri dosyası oluşturmak bu deneyin yerine geçmez.

In [ ]:
import importlib
import experiment
importlib.reload(experiment)
CSV, SOURCE_MODE = experiment.locate_csv(allow_download=False)
print("Veri kümesi:", experiment.DATASET_HANDLE)
print("Dosya:", CSV)
print("Boyut:", CSV.stat().st_size, "bayt")
print("SHA-256:", experiment.sha256_file(CSV))
print("Kaynak tespiti:", SOURCE_MODE)
print("Kaggle Input panelinde bu dosyanın belirtilen veri setine ait olduğunu kontrol edin.")

## 3. Üç modeli aynı ayrımda değerlendir
A: TF-IDF + LR; B: yalnızca 14 gösterge + LR; C: birleşik özellikler + LR.

Yaklaşık %60/%20/%20 eğitim/doğrulama/test; grup çakışması denetimi; C seçimi yalnızca doğrulamada; test eşiği 0,5. AP, trapez PR-AUC olarak sunulmaz.

In [ ]:
from datetime import datetime, timezone
OUTPUT = Path("results") / ("kaggle_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ"))
RESULT = experiment.run_experiment(CSV, OUTPUT, SOURCE_MODE,
    seed=42, max_features=50_000, max_chars=20_000, bootstrap=1000)
assert RESULT["manifest"]["status"] == "completed"
print("Gerçek deney çıktıları:", OUTPUT)

## 4. Ölçülen sonuçlar
Aşağıdaki hücreler, yalnızca yukarıdaki çalışma başarıyla tamamlanırsa veri gösterir. İyileşme yoksa bu da raporlanmalıdır.

In [ ]:
import pandas as pd
from IPython.display import display, Image
display(pd.DataFrame(RESULT["metrics"]))
print("Veri denetimi:", RESULT["audit"])
print("C - A, F1 farkı ve güven aralığı:", RESULT["comparison"])
display(Image(filename=str(OUTPUT / "f1_comparison.png")))

## 5. Deney kanıtlarını paketle
Arşivde ölçümler, veri özeti, ayrım manifesti ve gerçek test tahminleri vardır. Ham e-posta metinleri eklenmez. Tam metin ve sunuma kod yerine akış ve ölçülen sonuçlar aktarılır.

In [ ]:
import shutil
from IPython.display import FileLink
archive = shutil.make_archive(str(OUTPUT), "zip", root_dir=OUTPUT)
display(FileLink(archive))

## Sınırlılıklar
Tek ayrımda iç değerlendirme, gerçek hayata genelleme kanıtı değildir. Şablon gruplaması bütün yakın kopyaları yakalayamaz. Modern veya Türkçe e-postalarda dış doğrulama yapılmamaktadır. Veri kümesi spam ile oltalamayı birleştiriyorsa sonuç saf oltalama başarısı olarak adlandırılamaz. GitHub deposu henüz oluşturulmamıştır; temsili bir URL kullanılmamalıdır.